## Importación de librerias

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

## Importación de datos

In [2]:
customers = pd.read_csv('../data/raw/customers.csv')
orders = pd.read_csv('../data/raw/orders.csv')
payments = pd.read_csv('../data/raw/order-payments.csv')
items = pd.read_csv('../data/raw/order-items.csv')
reviews = pd.read_csv('../data/raw/order-reviews.csv')

## Overview y chequeo general de los datos

In [3]:
#breve overview del contenido de mis datos:
dataframes = { 
    'customers' : customers,
    'orders' : orders,
    'payments' : payments,
    'items' : items,
    'reviews' : reviews
}

for name, df in dataframes.items():
    profile = pd.DataFrame({
        'dtype' : df.dtypes,
        'cantidad nulos' : df.isnull().sum()
        })
    profile = profile.sort_values(by='cantidad nulos', ascending = False)

    print(f'\n====== {name.upper()} ======')
    print(profile)
    print('-' * 35)


====== CUSTOMERS ======
                          dtype  cantidad nulos
customer_id                 str               0
customer_unique_id          str               0
customer_zip_code_prefix  int64               0
customer_city               str               0
customer_state              str               0
-----------------------------------

====== ORDERS ======
                              dtype  cantidad nulos
order_delivered_customer_date   str            2965
order_delivered_carrier_date    str            1783
order_approved_at               str             160
order_id                        str               0
customer_id                     str               0
order_status                    str               0
order_purchase_timestamp        str               0
order_estimated_delivery_date   str               0
-----------------------------------

====== PAYMENTS ======
                        dtype  cantidad nulos
order_id                  str               0
payment_s

In [4]:
#chequeo de estructura
print(f'Orders unicas: {orders['order_id'].nunique() == len(orders)}')

#chequeo reviews duplicadas
review_duplicadas = reviews.duplicated(subset='order_id').sum()

review_orders_afectadas = reviews[reviews.duplicated(subset='order_id', keep=False)]['order_id'].nunique()

print(f'Reviews duplicadas: {review_duplicadas}')
print(f'Orders afectadas: {review_orders_afectadas} ({review_orders_afectadas / len(reviews) *100:.2f}%)')

#chequeo nulos
print(f'Delivered date nulos: {orders['order_delivered_customer_date'].isnull().mean():.2%}')
print(f'Review score nulos: {reviews['review_score'].isnull().mean():.2%}')

Orders unicas: True
Reviews duplicadas: 559
Orders afectadas: 555 (0.56%)
Delivered date nulos: 2.98%
Review score nulos: 0.00%


## Limpieza y transformación de los datos

In [5]:
#chequeo y formateo de las variables 'fecha' para correcta operativa
date_columns = [
    'order_purchase_timestamp', 
    'order_approved_at', 
    'order_delivered_carrier_date', 
    'order_delivered_customer_date', 
    'order_estimated_delivery_date'
]

for col in date_columns:
    orders[col] = pd.to_datetime(orders[col], errors='coerce')

items['shipping_limit_date'] = pd.to_datetime(items['shipping_limit_date'], errors='coerce')

reviews['review_creation_date'] = pd.to_datetime(reviews['review_creation_date'], errors='coerce')
reviews['review_answer_timestamp'] = pd.to_datetime(reviews['review_answer_timestamp'], errors='coerce')

In [6]:
#verifico orden cronológico de las columnas con formato 'fecha' y verifico que no haya entregas antes de realizada la compra
inconsistencias = orders[orders['order_delivered_customer_date'] < orders['order_purchase_timestamp']]

print(f"Registros con fechas ilógicas: {len(inconsistencias)}")


Registros con fechas ilógicas: 0


In [7]:
#chequeo el rango temporal de mi dataset
print(f"Inicio de datos: {orders['order_purchase_timestamp'].min()}")
print(f"Fin de datos: {orders['order_purchase_timestamp'].max()}")

Inicio de datos: 2016-09-04 21:15:19
Fin de datos: 2018-10-17 17:30:18


### Limpieza y transformación > Dataset Reviews

In [8]:
#confirmo reviews duplicadas
review_duplic = reviews.duplicated(subset='order_id', keep=False)
filas_duplic = reviews[review_duplic]

print(f'Filas reviews duplicadas: {len(filas_duplic)}')
print(f'Orders con reviews duplicadas: {filas_duplic['order_id'].nunique()}')
print("-" * 30)

Filas reviews duplicadas: 1114
Orders con reviews duplicadas: 555
------------------------------


In [9]:
#reviso qué hay dentro de las reviews duplicadas 
filas_duplic.groupby('order_id')['review_score'].nunique().value_counts()

review_score
1    346
2    209
Name: count, dtype: int64

#### Al constatar que son reviews de 1 y 2 estrellas (calificación baja) decido quedarme con la última review y conservar así la riqueza de esa información que podría tener utilidad en mi etapa de análisis.

In [10]:
#elimino reviews duplicadas conservando la más reciente
reviews_sin_duplicados = reviews.drop_duplicates(subset='order_id', keep='last')

In [11]:
#verifico lo anterior
print(f'Reviews duplicadas: {reviews_sin_duplicados['order_id'].duplicated().sum()}')

Reviews duplicadas: 0


In [12]:
#chequeo mi dataset de reviews 
profile_reviews = pd.DataFrame({
    'dtype': reviews.dtypes,
    'cantidad nulos': reviews.isnull().sum(),
    'porcentaje nulos': (reviews.isnull().sum() / len(reviews)) * 100
})

profile_reviews = profile_reviews.sort_values(by='cantidad nulos', ascending=False)

print(profile_reviews)

                                  dtype  cantidad nulos  porcentaje nulos
review_comment_title                str           88285            88.285
review_comment_message              str           58247            58.247
review_id                           str               0             0.000
order_id                            str               0             0.000
review_score                      int64               0             0.000
review_creation_date     datetime64[us]               0             0.000
review_answer_timestamp  datetime64[us]               0             0.000


### Como mi análisis no profundizará en las reviews (me limitaré a la calificación de las mismas), decido eliminar la columna "review_comment_title"

In [13]:
#elimino las columnas 'review_comment_title'
reviews = reviews.drop(columns='review_comment_title')

In [14]:
#verifico lo anterior
print(reviews.columns)

Index(['review_id', 'order_id', 'review_score', 'review_comment_message',
       'review_creation_date', 'review_answer_timestamp'],
      dtype='str')


### Limpieza y transformación > Dataset Orders

In [15]:
#chequeo las ordenes con status 'delivered' y nulos en 'order_delivered_customer_date' lo que sería un error de confirmación
nulo_error_entrega = (orders['order_status'] == 'delivered') & (orders['order_delivered_customer_date'].isnull())
casos_error = orders[nulo_error_entrega]

#chequeo los nulos que sí son coherentes
nulo_normal = (orders['order_status'].isin(['shipped', 'canceled', 'unavailable', 'invoiced', 'processing'])) & \
                   (orders['order_delivered_customer_date'].isnull())
casos_normales = orders[nulo_normal]

print(f'--- Diagnóstico de Integridad ---')
print(f'Pedidos "delivered" sin fecha (Inconsistencias): {len(casos_error)}')
print(f'Pedidos en tránsito/cancelados sin fecha (Normal): {len(casos_normales)}')
print(f'Total de nulos en order_delivered_customer_date: {orders['order_delivered_customer_date'].isnull().sum()}')

--- Diagnóstico de Integridad ---
Pedidos "delivered" sin fecha (Inconsistencias): 8
Pedidos en tránsito/cancelados sin fecha (Normal): 2950
Total de nulos en order_delivered_customer_date: 2965


In [16]:
#procedo a eliminar los pedidos "delivered" sin fecha
total_inicial = len(orders)

condicion_error = (orders['order_status'] == 'delivered') & (orders['order_delivered_customer_date'].isnull())

#aplico la limpieza utilizando el operador de negación (~), esto mantiene todo lo que no cumple la 'condicion_error'
orders = orders[~condicion_error].copy()

#verificación
total_final = len(orders)
eliminados = total_inicial - total_final

print(f"--- Resumen de Limpieza ---")
print(f"Registros inconsistentes eliminados: {eliminados}")
print(f"Registros restantes en 'orders': {total_final}")

--- Resumen de Limpieza ---
Registros inconsistentes eliminados: 8
Registros restantes en 'orders': 99433


In [17]:
#comprobación final para ver si quedan nulos con status 'delivered'
restantes_nulos_delivered = orders[(orders['order_status'] == 'delivered') & 
                                   (orders['order_delivered_customer_date'].isnull())]

print(f"Inconsistencias restantes: {len(restantes_nulos_delivered)}")

Inconsistencias restantes: 0


In [18]:
#analizo la consistencia de los nulos 'order_approved_at' (160 nulos) y 'order_delivered_carrier_date' (1783 nulos)
nulos_approved = orders[orders['order_approved_at'].isnull()]['order_status'].value_counts()

nulos_carrier = orders[orders['order_delivered_carrier_date'].isnull()]['order_status'].value_counts()

print('--- Estado de pedidos con Fecha de Aprobación NULA ---')
print(nulos_approved)

print('\n--- Estado de pedidos con Fecha de Transportista NULA ---')
print(nulos_carrier)

--- Estado de pedidos con Fecha de Aprobación NULA ---
order_status
canceled     141
delivered     14
created        5
Name: count, dtype: int64

--- Estado de pedidos con Fecha de Transportista NULA ---
order_status
unavailable    609
canceled       550
invoiced       314
processing     301
created          5
approved         2
delivered        1
Name: count, dtype: int64


In [19]:
#elimino los registros que son incosistentes como pedidos que figuran como 'entregado' pero la 'aprobación del pedido' es nula, o que la 'fecha de entrega al transportista' sea nula también.
inicial = len(orders)

#caso A: Entregados que no tienen fecha de aprobación
error_aprobacion = (orders['order_status'] == 'delivered') & (orders['order_approved_at'].isnull())

#caso B: Entregados que no tienen fecha de salida al transportista
error_carrier = (orders['order_status'] == 'delivered') & (orders['order_delivered_carrier_date'].isnull())

#aplico limpieza (eliminando ambos casos)
orders = orders[~(error_aprobacion | error_carrier)].copy()

print(f'Registros eliminados por inconsistencia logística: {inicial - len(orders)}')
print(f'Nuevos nulos en "order_approved_at": {orders['order_approved_at'].isnull().sum()}')
print(f'Nuevos nulos en "order_delivered_carrier_date": {orders['order_delivered_carrier_date'].isnull().sum()}')

Registros eliminados por inconsistencia logística: 15
Nuevos nulos en "order_approved_at": 146
Nuevos nulos en "order_delivered_carrier_date": 1781


#### Los nulos restantes son nulos "coherentes" a la lógica operativa del e-commerce. Por ej.: un pedido cancelado que no tiene una "order_approved", y un pedido en proceso/facturado no necesariamente debe estar en manos del transportista. Esto último ya corresponde a la eficiencia operativa que analizaré más adelante. 

### Antes de proceder al merge de mis datasets, hago una última verificación de nulos y formatos:

In [20]:
 #hago una última comprobación de todos los datasets (previo al merge)
dataframes = { 
    'customers' : customers,
    'orders' : orders,
    'payments' : payments,
    'items' : items,
    'reviews' : reviews
}

for name, df in dataframes.items():
    profile = pd.DataFrame({
        'dtype' : df.dtypes,
        'cantidad nulos' : df.isnull().sum()
        })
    profile = profile.sort_values(by='cantidad nulos', ascending = False)

    print(f'\n====== {name.upper()} ======')
    print(profile)
    print('-' * 35)


====== CUSTOMERS ======
                          dtype  cantidad nulos
customer_id                 str               0
customer_unique_id          str               0
customer_zip_code_prefix  int64               0
customer_city               str               0
customer_state              str               0
-----------------------------------

====== ORDERS ======
                                        dtype  cantidad nulos
order_delivered_customer_date  datetime64[us]            2957
order_delivered_carrier_date   datetime64[us]            1781
order_approved_at              datetime64[us]             146
order_id                                  str               0
customer_id                               str               0
order_status                              str               0
order_purchase_timestamp       datetime64[us]               0
order_estimated_delivery_date  datetime64[us]               0
-----------------------------------

====== PAYMENTS ======
           

## Conformo mi dataframe (uniendo los 5 datasets de mi input)

In [21]:
#merge de ordenes con clientes
df = orders.merge(customers, on='customer_id', how='left')

print(f'Paso 1 - Filas: {df.shape[0]}')

Paso 1 - Filas: 99418


In [22]:
#agrego items a mi dataframe
df = df.merge(items, on='order_id', how='left')

print(f'Paso 2 - Filas: {df.shape[0]}')

Paso 2 - Filas: 113401


In [23]:
#añado pagos y reviews
df = df.merge(payments, on='order_id', how='left')
df = df.merge(reviews, on='order_id', how='left')

print(f'Paso Final - Filas: {df.shape[0]}')

Paso Final - Filas: 119127


In [24]:
#chequeo duplicados que se pudieren generar con el merge
print(f'Duplicados totales: {df.duplicated().sum()}')

Duplicados totales: 0


### Breve overview de mi nuevo dataframe (df) sobre el cual haré el análisis

In [25]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 119127 entries, 0 to 119126
Data columns (total 27 columns):
 #   Column                         Non-Null Count   Dtype         
---  ------                         --------------   -----         
 0   order_id                       119127 non-null  str           
 1   customer_id                    119127 non-null  str           
 2   order_status                   119127 non-null  str           
 3   order_purchase_timestamp       119127 non-null  datetime64[us]
 4   order_approved_at              118965 non-null  datetime64[us]
 5   order_delivered_carrier_date   117043 non-null  datetime64[us]
 6   order_delivered_customer_date  115714 non-null  datetime64[us]
 7   order_estimated_delivery_date  119127 non-null  datetime64[us]
 8   customer_unique_id             119127 non-null  str           
 9   customer_zip_code_prefix       119127 non-null  int64         
 10  customer_city                  119127 non-null  str           
 11  customer_st

In [26]:
df.sample(10)

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_unique_id,customer_zip_code_prefix,...,freight_value,payment_sequential,payment_type,payment_installments,payment_value,review_id,review_score,review_comment_message,review_creation_date,review_answer_timestamp
67254,0f38051157b7ad1d4e0b6c5d20de6ae7,af1f25102280108074d2a310ffe4bd1e,delivered,2018-07-31 12:06:37,2018-08-01 02:35:12,2018-08-02 14:54:00,2018-08-06 16:14:38,2018-08-15,d8914408a1f960155489582d080bb52a,21941,...,6.28,1.0,boleto,1.0,142.65,0b675fa55c2d552f90a5b15c6b57654e,1,Não recebi até o momento o produto: porta lápi...,2018-08-07,2018-08-13 11:33:46
76858,55277930669682d227d5eedad65f60f8,5079d4b0bb65a49c39eda48508899a83,delivered,2017-09-24 13:38:28,2017-09-24 13:50:17,2017-09-25 20:22:20,2017-09-29 19:05:50,2017-10-25,0c8dd93583b64079a749cb0874a9b73c,35570,...,17.21,1.0,credit_card,1.0,127.11,7c258a557716de0f5fe1236bff8de0e6,5,NaN,2017-09-30,2017-10-02 22:40:34
79393,a48b334c5fecb11f072a3a98ae5d200f,a37642199abe998aeba2601b3f261e33,delivered,2017-12-28 11:05:22,2017-12-28 11:13:23,2017-12-28 21:13:59,2018-01-11 00:24:17,2018-01-24,4ecc3d1536d45f43dc367a3face29547,29102,...,14.76,1.0,credit_card,3.0,159.66,3e73551fbcd91f7cc5bc1eb8402a3c87,3,NaN,2018-01-11,2018-01-12 02:07:02
108989,99f10393ec3328b3b416a154afd5d714,b593c249903e4280399e991fdbe8d7a3,delivered,2017-07-19 23:51:41,2017-07-20 00:45:20,2017-07-26 11:07:51,2017-08-04 18:06:08,2017-08-17,afdd204d84a80bf0269cd6a84eb8c274,27288,...,17.67,1.0,credit_card,5.0,155.14,16814e4d2e1cb7da90f927f41cb1bb77,5,NaN,2017-08-05,2017-08-07 19:26:10
47730,4f0c82eab94eb2ea093382969dfe6949,66acec0b594d1e6da7e248d3172fcc43,delivered,2017-08-22 09:24:02,2017-08-24 03:15:28,2017-08-24 14:23:13,2017-08-25 18:42:02,2017-09-04,39d7ce28dfadb187458ae98f1e7e3857,3420,...,8.72,1.0,boleto,1.0,33.71,8dc989dcb5f125b9613c740c96955744,5,NaN,2017-08-26,2017-08-29 00:55:37
109125,50229fec371aedaedb230fa801638cfc,2866e688a5dc459c904ea765cae01f75,delivered,2018-01-22 20:41:50,2018-01-22 20:58:09,2018-01-23 19:04:52,2018-02-05 21:33:08,2018-02-16,1e7a2d6aa124d8f78af6e31b5e9cabad,87020,...,15.10,1.0,credit_card,3.0,35.00,43a715b2ff41fad40b5870df617f4d04,5,NaN,2018-02-06,2018-02-07 10:43:35
66284,0f6fb6d733b3a9e800c8aa92abf81584,fe790c996b162e9fb690b54ca141f061,delivered,2018-02-07 11:57:29,2018-02-07 12:10:38,2018-02-08 20:12:38,2018-02-10 09:56:43,2018-02-23,2445ba0a1c56ba4a3d4e202dc471aacc,3208,...,9.34,1.0,credit_card,1.0,138.48,9da9c6dcca338230d9a4c815efc681d9,4,NaN,2018-02-11,2018-02-15 22:37:57
24395,304c4a8ef20139552992b5377ef4a113,471084a5a72b0eebf59b306a9a125643,delivered,2017-08-14 17:29:46,2017-08-14 17:45:29,2017-08-15 17:10:23,2017-08-16 20:23:48,2017-08-29,2a18f594fb3541a8f06e5ac01cbea8f0,13320,...,7.78,1.0,credit_card,2.0,56.78,e5bc2f12575d179c6b73c3cb307adcfd,5,"Recebi o produto muito antes do prazo, gostei ...",2017-08-17,2017-08-20 14:34:36
118408,04daee30107ba31302f8528918ba7f37,58723c16d88f82622b9cd3888d90dd55,delivered,2018-02-21 11:19:32,2018-02-23 02:40:24,2018-02-24 12:56:32,2018-03-15 12:42:49,2018-03-19,9fe9abefabab3110c6acb617f19159ac,87305,...,15.10,1.0,boleto,1.0,40.00,5162f0449c74538b990fad9f35551230,5,NaN,2018-03-16,2018-03-16 16:58:00
18781,5999d0076050549364f1df892e1aeaa5,28c0c209da1b812e1f01a311024a61c5,delivered,2018-03-22 10:14:00,2018-03-23 10:08:45,2018-03-28 01:52:15,2018-04-02 19:38:42,2018-04-13,56c8638e7c058b98aae6d74d2dd6ea23,37800,...,22.93,1.0,credit_card,1.0,43.82,9efa3616388f3702b08359dc21ae7ecd,5,NaN,2018-04-03,2018-04-04 00:53:42


In [27]:
#chequeo los nulos de mi nuevo dataframe
df.isnull().sum()

order_id                             0
customer_id                          0
order_status                         0
order_purchase_timestamp             0
order_approved_at                  162
order_delivered_carrier_date      2084
order_delivered_customer_date     3413
order_estimated_delivery_date        0
customer_unique_id                   0
customer_zip_code_prefix             0
customer_city                        0
customer_state                       0
order_item_id                      833
product_id                         833
seller_id                          833
shipping_limit_date                833
price                              833
freight_value                      833
payment_sequential                   3
payment_type                         3
payment_installments                 3
payment_value                        3
review_id                            0
review_score                         0
review_comment_message           67890
review_creation_date     

#### Hago uso de la Extensión "Data Wrangler" de Microsoft para visualizar de forma práctica el dataframe y hacer uso de filtros múltiples. De esta forma noto que tengo 3 registros nulos en todas las columnas referentes al pago y 833 registros nulos referentes a items.

### Exploramos: los 3 nulos referidos al "payment"

In [28]:
#determino las columnas a revisar
payment_cols = ['payment_sequential', 'payment_type', 'payment_installments', 'payment_value']

#filtro las filas donde esas columnas tengan nulos
nulos_pago = df[df[payment_cols].isnull().all(axis=1)]

print(f'Se encontraron {len(nulos_pago)} filas con información de pago inexistente.')
nulos_pago[['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'review_comment_message'] + payment_cols]



Se encontraron 3 filas con información de pago inexistente.


,order_id,customer_id,order_status,order_purchase_timestamp,review_comment_message,payment_sequential,payment_type,payment_installments,payment_value
36860,bfbd0f9bdef84302105ad712db648a6c,86dc2ffce2dfff336de2f386a786e574,delivered,2016-09-15 12:16:38,nao recebi o produto e nem resposta da empresa,NaN,NaN,NaN,NaN
36861,bfbd0f9bdef84302105ad712db648a6c,86dc2ffce2dfff336de2f386a786e574,delivered,2016-09-15 12:16:38,nao recebi o produto e nem resposta da empresa,NaN,NaN,NaN,NaN
36862,bfbd0f9bdef84302105ad712db648a6c,86dc2ffce2dfff336de2f386a786e574,delivered,2016-09-15 12:16:38,nao recebi o produto e nem resposta da empresa,NaN,NaN,NaN,NaN


#### Vemos que se trata de la misma orden, sin datos del pago y el cliente comenta no recibir el producto, probablemente hubo un fallo técnico en el pago. Procedo a eliminar las 3 filas para un dataframe más limpio y reportaré el hallazgo en el informe.

In [29]:
#elimino las filas mencionadas 
df = df.dropna(subset=payment_cols, how='all').copy()

print(f"Limpieza completada. Nuevo tamaño del dataset: {df.shape}")

Limpieza completada. Nuevo tamaño del dataset: (119124, 27)


### Exploramos: los 162 pedidos sin aprobación (nulos de "order_approved_at")

In [ ]:
#filtro el datafrane para los casos sin fecha de aprobación
df_nulos_aprobacion = df[df['order_approved_at'].isnull()]

#analizo la distribución del método de pago
print("--- Métodos de pago en pedidos NO aprobados ---")
print(df_nulos_aprobacion['payment_type'].value_counts(dropna=False))

#analizo el estado de los pedidos
print("\n--- Estado de los pedidos NO aprobados ---")
print(df_nulos_aprobacion['order_status'].value_counts())

--- Métodos de pago en pedidos NO aprobados ---
payment_type
voucher        86
credit_card    57
boleto         16
not_defined     3
Name: count, dtype: int64

--- Estados de los pedidos con aprobación nula ---
order_status
canceled    157
created       5
Name: count, dtype: int64


In [31]:
#hago un diagnóstico para los 162 pedidos sin orden de aprobación incluyendo la review y la review_score

nulos_aprobacion = df[df['order_approved_at'].isnull()]

cols_diagnostico = [
    'order_id', 
    'order_status', 
    'order_purchase_timestamp', 
    'payment_type', 
    'review_score', 
    'review_comment_message'
]

print(f'Para los {len(nulos_aprobacion)} pedidos sin fecha de aprobación visualizo lo siguente:')
nulos_aprobacion[cols_diagnostico].head(10)

Para los 162 pedidos sin fecha de aprobación visualizo lo siguente:


,order_id,order_status,order_purchase_timestamp,payment_type,review_score,review_comment_message
1362,00b1cb0320190ca0daa2c88b35206009,canceled,2018-08-28 15:26:39,not_defined,1,Comprei dois fones de ouvido com valor de R$ 5...
2142,ed3efbd3a87bea76c2812c66a0b32219,canceled,2018-09-20 13:54:16,voucher,2,O produto veio com defeito ele não liga não fu...
2232,df8282afe61008dc26c6c31011474d02,canceled,2017-03-04 12:14:30,boleto,3,Razoável
2430,8d4c637f1accf7a88a4555f02741e606,canceled,2018-08-29 16:27:49,voucher,5,NaN
2587,7a9d4c7f9b068337875b95465330f2fc,canceled,2017-05-01 16:12:39,credit_card,5,NaN
3642,ddaec6fff982b13e7e048b627a11d6da,canceled,2016-10-04 19:41:32,boleto,1,"Nunca chegou o produto, deram uma excusa e tro..."
3686,5290c34bd38a8a095b885f13958db1e1,canceled,2018-08-21 10:25:18,voucher,3,NaN
4354,03310aa823a66056268a3bab36e827fb,canceled,2018-08-07 16:33:59,voucher,1,Entrega do produto diferente do solicitado\r\n...
5169,4c8b9947280829d0a8b7e81cc249b875,canceled,2018-08-09 14:54:47,voucher,1,"Lentes de contato são positivas, para hipermet..."
5820,b13ea375fe9c728832688264638f84cf,canceled,2018-08-22 18:52:29,voucher,1,NaN


#### Con este diagnóstico y explorando con DataWrangler, veo que el 99% de estos pedidos sin aprobación están cancelados, pero muchos sí han sido entregados (con errores) y otros tantos no fueron entregados. Entiendo que esto se debió a algún fallo en el pago o el procesamiento de la orden. Conservaré los datos para que se refleje en el EDA.

In [32]:
#creo una flag indicativa de estos nulos para futuras operaciones
df['approval_status_flag'] = 'approved'

#etiqueto según los casos que tengo en dichos nulos (canceled y created) y asigno "error_procesamiento"
df.loc[df['order_approved_at'].isnull() & df['order_status'].isin(['canceled', 'created']), 'approval_status_flag'] = 'error_procesamiento'

### Exploramos: los 3413 nulos de "order_delivered_customer_date"

In [33]:
#filtro los nulos
nulos_logistica = df[df['order_delivered_customer_date'].isnull()]

#cruzo con no faltantes de 'order_delivered_carrier_date' para ver pedidos sin fecha de entrega al cliente pero que si fueron recibidos por transportista (paquetes perdidos)
nulos_logistica['entregado_al_correo'] = nulos_logistica['order_delivered_carrier_date'].notnull()

print("--- Análisis de Nulos de Entrega ---")
print(nulos_logistica.groupby(['order_status', 'entregado_al_correo']).size().unstack(fill_value=0))

--- Análisis de Nulos de Entrega ---
entregado_al_correo  False  True 
order_status                     
approved                 3      0
canceled               670     73
created                  5      0
invoiced               378      0
processing             376      0
shipped                  0   1256
unavailable            652      0


#### Luego del análisis, destaco dos puntos a profundizar más adelante: 73 pedidos cancelados posteriormente a la entrega al transportista. Y 1256 en curso de envío (revisaré luego si llevan mucho tiempo en ese estado).

### Exploramos: los 833 nulos referidos a "items"

In [34]:
#determino los registros con ítems nulos
nulos_items = df[df['order_item_id'].isnull()]

#analizo el estado de estos pedidos
status_summary = nulos_items['order_status'].value_counts()

print("--- Distribución por Estado del Pedido ---")
print(status_summary)

--- Distribución por Estado del Pedido ---
order_status
unavailable    645
canceled       180
created          5
invoiced         2
shipped          1
Name: count, dtype: int64


#### Al constatar que la mayoría son nulos por falta de stock (unavailable) y operaciones no concretadas (cancelled) decido mantenerlos para un posterior análisis de eficiencia operativa.

In [35]:
#reviso la cantidad de nulos en las restantes columnas de ordenes (con 'order_item_id' nulo)
nulos_ordenes = df[df['order_item_id'].isnull()]

cols_ordenes = [
    'product_id', 
    'seller_id', 
    'shipping_limit_date', 
    'price', 
    'freight_value'
]

print("Conteo de nulos para el subconjunto de 'order_item_id' nulo:")
print(nulos_ordenes[cols_ordenes].isnull().sum())

Conteo de nulos para el subconjunto de 'order_item_id' nulo:
product_id             833
seller_id              833
shipping_limit_date    833
price                  833
freight_value          833
dtype: int64


#### Eliminaré las columnas "product_id" y "seller_id" dado que no me aportan a mi análisis. Imputaré los nulos de "order_item_id" y "price" con ceros dado que esas ordenes no tienen artículos. Para "shipping_limit_date" conservaré los nulos y respecto a la columna del costo del flete "freight_value" observé (filtrando con DataWrangler) que tiene 390 ceros por lo que imputaré los nulos con -1 preservar su integridad y no sesgar el análisis.

In [36]:
#elimino columnas de producto y vendedor 
df = df.drop(columns=['product_id', 'seller_id'])

In [37]:
#imputo los nulos de la cantidad de items
df['order_item_id'] = df['order_item_id'].fillna(0).astype('Int64')

#para el precio también imputo con ceros
df['price'] = df['price'].fillna(0.0)

#para el costo de envío (freight_value) imputo con -1 para conservar los pedidos sin costo de envío
df['freight_value'] = df['freight_value'].fillna(-1.0)

#### Creo una nueva columna (booleana) como flag indicativa de las ventas que sí se han concretado. Su función es descartar fácilmente, en un posterior análisis, los ceros que he sustituido.

In [38]:
#bandera que indicará las ventas efectivas
df['effective_sale'] = df['price'] > 0

In [39]:
#chequeo ahora los nulos de mi dataframe
df.isnull().sum()

order_id                             0
customer_id                          0
order_status                         0
order_purchase_timestamp             0
order_approved_at                  162
order_delivered_carrier_date      2084
order_delivered_customer_date     3413
order_estimated_delivery_date        0
customer_unique_id                   0
customer_zip_code_prefix             0
customer_city                        0
customer_state                       0
order_item_id                        0
shipping_limit_date                833
price                                0
freight_value                        0
payment_sequential                   0
payment_type                         0
payment_installments                 0
payment_value                        0
review_id                            0
review_score                         0
review_comment_message           67890
review_creation_date                 0
review_answer_timestamp              0
approval_status_flag     

In [40]:
#chequeo los formatos de las variables
df.info()

<class 'pandas.DataFrame'>
Index: 119124 entries, 0 to 119126
Data columns (total 27 columns):
 #   Column                         Non-Null Count   Dtype         
---  ------                         --------------   -----         
 0   order_id                       119124 non-null  str           
 1   customer_id                    119124 non-null  str           
 2   order_status                   119124 non-null  str           
 3   order_purchase_timestamp       119124 non-null  datetime64[us]
 4   order_approved_at              118962 non-null  datetime64[us]
 5   order_delivered_carrier_date   117040 non-null  datetime64[us]
 6   order_delivered_customer_date  115711 non-null  datetime64[us]
 7   order_estimated_delivery_date  119124 non-null  datetime64[us]
 8   customer_unique_id             119124 non-null  str           
 9   customer_zip_code_prefix       119124 non-null  int64         
 10  customer_city                  119124 non-null  str           
 11  customer_state  

#### Elimino más columnas que no son de interés para mi análisis: 
#### •	customer_zip_code_prefix
#### •	customer_city
#### •	payment_sequential
#### •	review_answer_timestamp

In [41]:
cols_a_eliminar = [
    'customer_zip_code_prefix', 
    'customer_city', 
    'payment_sequential', 
    'review_answer_timestamp'
]

df = df.drop(columns=cols_a_eliminar)

print(f'Total columnas del dataframe: {len(df.columns)}')

Total columnas del dataframe: 23


#### Transformo formato floats de "payment_installments" a Int64.

In [42]:
#transformo a Int64
df['payment_installments'] = df['payment_installments'].astype('Int64')

#verifico formato
print(f"Formato actual: {df['payment_installments'].dtype}")

Formato actual: Int64


In [43]:
#chequeo si existen pagos con Nro de cuotas = 0, lo que sería técnicamente incorrecto
cuotas_cero = df[df['payment_installments'] == 0]

print(f'Cantidad de órdenes con 0 cuotas: {len(cuotas_cero)}')


Cantidad de órdenes con 0 cuotas: 3


In [44]:
#dado que encuentro solo 3 registros con cuotas = 0 (que probablemente haya sido un error en la pasarela de pagos) decido eliminarlos
df = df[df['payment_installments'] != 0]

print(f'Nuevo total de filas: {len(df)}')

Nuevo total de filas: 119121


In [ ]:
#chequeo nuevamente mi dataframe
print(f'---- ESTRUCTURA DEL DATAFRAME ----')
print(f'Total de registros (filas): {df.shape[0]}')
print(f'Total de variables (columnas): {df.shape[1]}')
print('\n---- NULOS POR VARIABLES (en %) ----')
df.isnull().sum() / len(df)*100

--- ESTRUCTURA DEL DATAFRAME ---
Total de registros (filas): 119121
Total de variables (columnas): 23

---- NULOS POR VARIABLES (en %) ----


order_id                          0.000000
customer_id                       0.000000
order_status                      0.000000
order_purchase_timestamp          0.000000
order_approved_at                 0.135996
order_delivered_carrier_date      1.749482
order_delivered_customer_date     2.865154
order_estimated_delivery_date     0.000000
customer_unique_id                0.000000
customer_state                    0.000000
order_item_id                     0.000000
shipping_limit_date               0.699289
price                             0.000000
freight_value                     0.000000
payment_type                      0.000000
payment_installments              0.000000
payment_value                     0.000000
review_id                         0.000000
review_score                      0.000000
review_comment_message           56.989951
review_creation_date              0.000000
approval_status_flag              0.000000
effective_sale                    0.000000
dtype: floa

#### Conservo la variable de reseña (a pesar de tener 56% de nulos) dado que aportará utilidad en la siguiente etapa de análisis (especialmente para determinar posibles errores en la compra, demoras en el envío, etc).

### Guardamos el dataframe con la limpieza y transformación de los datos:

In [48]:
df.to_csv('../data/output/ecommerce-limpio.csv', index=False)